# Lab 1 exploratory notes

Logic lives in `clean.py`. This notebook is the narrative + light checks.
Figures are written to `../report/exploratory_figures/`.

In [ ]:
import pandas as pd
from IPython.display import display

import clean as cl

df = cl.load_train()
print(df.shape)
print("figures go to:", cl.FIG_DIR)

## 0) Train / test split (before cleaning)

Patient-level **80/20** split, stratified on `PosIntFinal`, created by `split.py`.
- train: `../data/splits/train.csv` — use this for all exploration and cleaning
- test: `../data/splits/test.csv` — **frozen**, do not load until final evaluation
- 20 patients with blank `PosIntFinal` are in neither set

Re-run with: `python split.py`

## 1) Missingness heatmap (blank vs 91/92)

First look at proportions of blank and sentinel-coded cells by CTDone and age band.

In [ ]:
cl.plot_missingness_heatmap(df, show=True)

A lot of variables have many 91/92 values. Dropping all those rows is not realistic —
especially since 91 means pre-verbal. Early ideas were: (1) drop rows, (2) impute averages,
(3) recode to NaN. Diagnostics below show why those need rethinking.

## 2) What is actually in 91/92?

Can't decide whether child columns are worth keeping until we see whether 92 is missing
or just skip logic.

In [ ]:
audit, coded_cols, real_valued = cl.sentinel_audit(df)
print(f"{len(coded_cols)} columns use 91/92 as codes")
print(f"but 91 only ever appears in: {[c for c in coded_cols if df[c].eq(91).any()]}")
display(audit.head(10))

print(f"!! 91/92 are REAL VALUES here, never recode: {real_valued}")
for c in real_valued:
    print(f"   {c}: range {df[c].min()}-{df[c].max()}")
    print(f"   counts near the codes: {df[c].value_counts().reindex(range(89, 95)).to_dict()}")

In [ ]:
gates = cl.find_gates(df, coded_cols)
n_det = int(gates.deterministic.sum())
print(f"{n_det} of {len(gates)} columns have their 92s FULLY predicted by a single parent")
print(gates[gates.deterministic].groupby("parent_or_closest")["child"].apply(list).to_string())
print("\nno parent perfectly explains these:")
display(gates[~gates.deterministic])

#### Charting the gating structure

Skip logic as a **tree**, plus the **information content** of each gated block.

In [ ]:
cl.plot_gating_tree(df, gates=gates, show=True)
cl.plot_gated_block_content(df, gates=gates, show=True)

#### The one exception — level-2 codes mean different things

`SFxPalp=2` ("unclear") gates the follow-up off; `LOCSeparate=2` ("suspected") does not.

In [ ]:
print("SFxPalp -> SFxPalpDepress")
display(pd.crosstab(df.SFxPalp.fillna("NaN"), df.SFxPalpDepress.fillna("NaN"), dropna=False))
print("\nLOCSeparate -> LocLen")
display(pd.crosstab(df.LOCSeparate.fillna("NaN"), df.LocLen.fillna("NaN"), dropna=False))

## 3) True blanks + outcomes

In [ ]:
blanks = (df.drop(columns=list(cl.ID_COLS)).isna().mean() * 100).sort_values(ascending=False)
display(blanks.head(12).round(1).to_frame("pct_blank"))

d, e = df.Dizzy.isna(), df.Ethnicity.isna()
lift = (d & e).mean() / (d.mean() * e.mean())
print(f"Dizzy/Ethnicity joint lift vs independence: {lift:.3f}")

display(
    pd.DataFrame(
        {
            "n_blank": df[cl.OUTCOMES].isna().sum(),
            "n_92": df[cl.OUTCOMES].eq(92).sum(),
            "n_positive": df[cl.OUTCOMES].eq(1).sum(),
            "pct_positive": (df[cl.OUTCOMES].eq(1).mean() * 100).round(2),
        }
    )
)

## 4) Verbal / preverbal cohorts

Preverbal = `91` on either `HA_verb` or `Amnesia_verb`. Keep no-CT patients (outcome known).

In [ ]:
cohorts = cl.build_cohorts(df)
df_labelled = cohorts["df_labelled"]
preverbal = cohorts["preverbal"]
df_separated_nonverbal = cohorts["df_separated_nonverbal"]
df_separated_verbal = cohorts["df_separated_verbal"]
label = cohorts["label"]

print(f"labelled {len(df_labelled)}; preverbal {len(df_separated_nonverbal)}, verbal {len(df_separated_verbal)}")
display(pd.crosstab(label, df_labelled.AgeTwoPlus))
display(
    pd.DataFrame(
        {
            "n": [len(df_separated_nonverbal), len(df_separated_verbal)],
            "outcome rate %": [
                round(df_separated_nonverbal.PosIntFinal.mean() * 100, 2),
                round(df_separated_verbal.PosIntFinal.mean() * 100, 2),
            ],
            "CT done %": [
                round(df_separated_nonverbal.CTDone.mean() * 100, 1),
                round(df_separated_verbal.CTDone.mean() * 100, 1),
            ],
        },
        index=["preverbal", "verbal"],
    )
)

In [ ]:
core_cols = cl.core_columns(df, gates)
print(f"{len(df.columns)} cols - gated children = {len(core_cols)} informative")
print(sorted(core_cols))


def modal_share(d, cols):
    return pd.Series({c: d[c].value_counts(dropna=False).iloc[0] / len(d) * 100 for c in cols})


share = pd.DataFrame(
    {
        "preverbal": modal_share(df_separated_nonverbal, core_cols),
        "verbal": modal_share(df_separated_verbal, core_cols),
    }
)
share["gap"] = (share.preverbal - share.verbal).abs()
display(share.sort_values("gap", ascending=False).head(10).round(1))
print("effectively constant in verbal only:",
      sorted(share.index[(share.verbal > 99) & (share.preverbal <= 99)]))

In [ ]:
d = df_labelled.Dizzy.isna()
display(pd.crosstab(label, d.map({True: "Dizzy blank", False: "Dizzy answered"})))
print(f"{(d & preverbal).sum() / d.sum() * 100:.1f}% of Dizzy blanks are preverbal")
print(f"blank rate: preverbal {d[preverbal].mean()*100:.1f}% vs verbal {d[~preverbal].mean()*100:.1f}%")

e = df_labelled.Ethnicity.isna()
print(f"Ethnicity: preverbal {e[preverbal].mean()*100:.1f}% vs verbal {e[~preverbal].mean()*100:.1f}%")

### re-exploring post-split

**~45 informative columns** after dropping recoverable gated children.
Preverbal kids have ~2.4× outcome rate while being CT'd less often.
`Dizzy` blanks are mostly structural (preverbal); `Ethnicity` is real missingness.

## 5) Availability heatmap (after diagnostics)

Informative predictors only; top = blank|91|92, bottom = true blank.

In [ ]:
cl.plot_availability_heatmap(df, gates=gates, cohorts=cohorts, show=True)

Very little true missingness among informative predictors. The main 91/92 mass that remains
visible is structural (e.g. `SFxPalpDepress`).

All figures also live under `../report/exploratory_figures/` (PDF). Regenerate everything with:

```bash
python clean.py
```